In [ ]:
# Install the Python packages required by this notebook.
%pip install -q duckdb pyarrow scikit-learn xgboost scipy joblib

In [ ]:
# Mount Google Drive so this notebook can access the private MIMIC-IV data and derived files.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Define the MIMIC-IV folders and the shared derived-data folder used by all notebooks.
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Early Acute Kidney Injury Prediction + Production Monitoring/data")
HOSP_DIR = DATA_ROOT / "hosp"
ICU_DIR = DATA_ROOT / "icu"
DERIVED_DIR = DATA_ROOT / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the libraries used for preprocessing, temporal splitting, and model training.
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

In [ ]:
# Load the engineered modeling table created by Notebook 03.
ml_dataset = pd.read_parquet(DERIVED_DIR / "ml_dataset.parquet")

print("Rows:", f"{len(ml_dataset):,}")
print("Columns:", ml_dataset.shape[1])
print("Positive rate:", f"{ml_dataset['target'].mean():.3%}")

In [ ]:
# Split the cohort into earlier training, intermediate validation, and later test periods using anchor-year groups.
TRAIN_GROUPS = {"2008 - 2010", "2011 - 2013", "2014 - 2016"}
VALIDATION_GROUPS = {"2017 - 2019"}
TEST_GROUPS = {"2020 - 2022"}

train_df = ml_dataset[
    ml_dataset["anchor_year_group"].isin(TRAIN_GROUPS)
].copy()

validation_df = ml_dataset[
    ml_dataset["anchor_year_group"].isin(VALIDATION_GROUPS)
].copy()

test_df = ml_dataset[
    ml_dataset["anchor_year_group"].isin(TEST_GROUPS)
].copy()

print("Train:", train_df.shape, f"positive={train_df['target'].mean():.3%}")
print("Validation:", validation_df.shape, f"positive={validation_df['target'].mean():.3%}")
print("Test:", test_df.shape, f"positive={test_df['target'].mean():.3%}")

In [ ]:
# Define the model input columns while keeping identifiers, race, target, and time-group metadata out of training.
EXCLUDED_COLUMNS = {
    "subject_id",
    "hadm_id",
    "stay_id",
    "target",
    "anchor_year_group",
    "race",
}

CATEGORICAL_COLUMNS = ["gender", "first_careunit"]

feature_columns = [
    column
    for column in ml_dataset.columns
    if column not in EXCLUDED_COLUMNS
]

categorical_columns = [
    column
    for column in CATEGORICAL_COLUMNS
    if column in feature_columns
]

numeric_columns = [
    column
    for column in feature_columns
    if column not in categorical_columns
]

print("Model features:", len(feature_columns))
print("Numeric features:", len(numeric_columns))
print("Categorical features:", len(categorical_columns))

In [ ]:
# Build reusable preprocessing pipelines for numeric imputation, scaling, and categorical one-hot encoding.
def build_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
    )

In [ ]:
# Train an interpretable logistic-regression baseline with class balancing.
X_train = train_df[feature_columns]
y_train = train_df["target"].astype(int)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)
print("Logistic regression trained.")

In [ ]:
# Train the main XGBoost classifier using a class-weight ratio derived from the training cohort.
positive_count = max(int(y_train.sum()), 1)
negative_count = max(int((1 - y_train).sum()), 1)
scale_pos_weight = negative_count / positive_count

xgboost_model = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "model",
            XGBClassifier(
                n_estimators=350,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

xgboost_model.fit(X_train, y_train)
print("XGBoost trained.")

In [ ]:
# Save both trained models, the exact feature list, and the temporal data splits for reproducible evaluation.
MODEL_DIR = DERIVED_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    logistic_model,
    MODEL_DIR / "logistic_regression.joblib",
)
joblib.dump(
    xgboost_model,
    MODEL_DIR / "xgboost.joblib",
)

with open(MODEL_DIR / "feature_columns.json", "w") as file:
    json.dump(feature_columns, file, indent=2)

train_df.to_parquet(DERIVED_DIR / "train.parquet", index=False)
validation_df.to_parquet(DERIVED_DIR / "validation.parquet", index=False)
test_df.to_parquet(DERIVED_DIR / "test.parquet", index=False)

print("Saved models and temporal splits.")